# HybridRAG with Medical Textbook — AlzRAGBench

**Part of the AlzRAGBench project** — a controlled ablation study comparing three Retrieval-Augmented Generation (RAG) architectures on an Alzheimer's disease knowledge corpus sourced from **medical textbooks, PubMed abstracts, and Wikipedia articles**.

This notebook uses **Gemini Flash Lite** (via Google GenAI API) as the answer generator — a standard autoregressive LLM, in contrast to the diffusion-based generator used in the companion Diffusion AlzRAGBench folder.

---

## What this notebook does

1. **Loads and visualizes** the hand-curated Alzheimer's disease Knowledge Graph (54 nodes, 89 edges) built from 31 source documents (20 PubMed abstracts + 10 Wikipedia articles + 1 StatPearls textbook chapter).
2. **Builds a VectorRAG pipeline** using FAISS and `all-MiniLM-L6-v2` embeddings over 573 text chunks extracted from the same corpus.
3. **Builds a GraphRAG pipeline** using NetworkX over the same knowledge graph, retrieving entity neighborhoods relevant to each query.
4. **Combines both into a HybridRAG pipeline** that merges textbook chunk context with structured graph context before generating an answer.
5. **Evaluates all three methods** (VectorRAG, GraphRAG, HybridRAG) head-to-head on a 30-question benchmark set, scoring with ROUGE-L and semantic cosine similarity.
6. **Visualizes results** with grouped bar charts and per-question score plots.

## Key differentiator vs other AlzRAGBench variants

The VectorRAG corpus here is built from **medical textbook content** (StatPearls) alongside PubMed and Wikipedia, making it richer in clinical detail than the article-only corpus used in the Medical Articles variant. This tests whether deeper textbook grounding improves VectorRAG's ability to answer clinical and mechanistic questions.

## 1 — Install Dependencies

In [ ]:
# Run once to install all required packages
# !pip install -q faiss-cpu sentence-transformers networkx pandas numpy matplotlib rouge-score scikit-learn google-genai python-dotenv pyvis

## 2 — Imports and Paths

In [ ]:
import os
import sys
import json
import time
import pickle
import warnings
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from google import genai

warnings.filterwarnings("ignore")

# ── Paths (relative to notebook location) ─────────────────────────────────────
NOTEBOOK_DIR  = Path(".").resolve()
BASE_DIR      = NOTEBOOK_DIR.parent

NODES_CSV     = BASE_DIR / "Dataset" / "Knowledge graph" / "nodes.csv"
EDGES_CSV     = BASE_DIR / "Dataset" / "Knowledge graph" / "edges.csv"
CHUNKS_PATH   = BASE_DIR / "Dataset" / "chunking"        / "_all_chunks.json"
EVAL_PATH     = BASE_DIR / "Dataset" / "Evaluation"      / "eval_dataset.json"
VECTOR_DIR    = BASE_DIR / "vector_output"
RESULTS_DIR   = BASE_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

# ── Gemini API key ─────────────────────────────────────────────────────────────
load_dotenv(BASE_DIR / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"

# ── Load evaluation questions ──────────────────────────────────────────────────
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

questions = eval_data["questions"]
dist = eval_data["distribution_by_target_method"]

print(f"Loaded {len(questions)} evaluation questions")
print(f"Distribution: {dist}")
print(f"\nPaths OK:")
print(f"  nodes.csv   : {NODES_CSV.exists()}")
print(f"  edges.csv   : {EDGES_CSV.exists()}")
print(f"  chunks.json : {CHUNKS_PATH.exists()}")

## 3 — Knowledge Graph: Load and Visualize

The knowledge graph was hand-curated from all 31 source documents. Each edge is grounded in at least one source article. We use **NetworkX** — with only 54 nodes and 89 edges, a full graph database (e.g., Neo4j) would be unnecessary overhead.

In [ ]:
# ── Load nodes and edges ───────────────────────────────────────────────────────
nodes_df = pd.read_csv(NODES_CSV)
edges_df = pd.read_csv(EDGES_CSV)

print(f"Nodes: {len(nodes_df)}")
print(f"Edges: {len(edges_df)}")
print(f"\nNode types: {nodes_df['type:LABEL'].value_counts().to_dict()}")
nodes_df.head()

In [ ]:
# ── Build NetworkX graph ───────────────────────────────────────────────────────
G = nx.Graph()

for _, row in nodes_df.iterrows():
    G.add_node(row["nodeId:ID"],
               name=row["name"],
               label=row["type:LABEL"],
               description=row["description"])

for _, row in edges_df.iterrows():
    G.add_edge(row[":START_ID"], row[":END_ID"],
               relation=row[":TYPE"],
               evidence=row["evidence"])

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected:    {nx.is_connected(G)}")

# Top 5 hub nodes by degree
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop-5 hubs (by degree):")
for node_id, deg in degree_sorted:
    name = G.nodes[node_id].get("name", node_id)
    print(f"  {name:<30} degree={deg}")

In [ ]:
# ── Static Visualization ───────────────────────────────────────────────────────
TYPE_COLORS = {
    "Disease":      "#E05C5C",
    "Drug":         "#5C8BE0",
    "Gene":         "#5CE07A",
    "GeneVariant":  "#B45CE0",
    "Protein":      "#E0A85C",
    "RiskFactor":   "#5CCFE0",
    "Biomarker":    "#E0D85C",
    "Mechanism":    "#E07ABB",
    "Pathology":    "#A0A0A0",
}

node_colors = [
    TYPE_COLORS.get(G.nodes[n].get("label", ""), "#cccccc")
    for n in G.nodes()
]

node_sizes = [300 + G.degree(n) * 80 for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 11))
pos = nx.spring_layout(G, seed=42, k=2.2)

nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color="#aaaaaa", ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       alpha=0.9, ax=ax)

labels = {n: G.nodes[n].get("name", n) for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=7, ax=ax)

patches = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()]
ax.legend(handles=patches, loc="upper left", fontsize=8, title="Entity Type")
ax.set_title("Alzheimer's Disease Knowledge Graph\n(node size ∝ degree)",
             fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "knowledge_graph_static.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Results/knowledge_graph_static.png")

In [ ]:
# ── Interactive Visualization (pyvis) ─────────────────────────────────────────
from pyvis.network import Network
from IPython.display import IFrame, display

net = Network(height="600px", width="100%", bgcolor="#1a1a2e", font_color="white")
net.barnes_hut()

for node_id in G.nodes():
    data  = G.nodes[node_id]
    color = TYPE_COLORS.get(data.get("label", ""), "#cccccc")
    net.add_node(str(node_id),
                 label=data.get("name", str(node_id)),
                 title=f"{data.get('label','')}\n{data.get('description','')}",
                 color=color,
                 size=10 + G.degree(node_id) * 3)

for u, v, data in G.edges(data=True):
    net.add_edge(str(u), str(v), title=data.get("relation", ""),
                 color="#555555")

html_path = str(RESULTS_DIR / "knowledge_graph_interactive.html")
net.save_graph(html_path)
display(IFrame(html_path, width="100%", height="620px"))
print("Saved: Results/knowledge_graph_interactive.html")

## 4 — VectorRAG Pipeline

We embed all text chunks using `all-MiniLM-L6-v2` and store them in a **FAISS** flat L2 index. At query time, the query is embedded and the top-5 nearest chunks are retrieved as context.

The corpus consists of 573 sentence-boundary-aware chunks (~220 words each, ~40-word overlap) extracted from:
- 1 StatPearls textbook chapter on Alzheimer's disease ← **key differentiator**
- 20 PubMed abstracts
- 10 Wikipedia articles

In [ ]:
# ── Load chunks ────────────────────────────────────────────────────────────────
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts    = [c["text"] for c in chunks]
metadata = [{"chunk_id":   c["chunk_id"],
             "article_id": c["article_id"],
             "title":      c["metadata"].get("title", "")} for c in chunks]

print(f"Loaded {len(texts)} chunks")

# Show source breakdown
from collections import Counter
sources = Counter(m["article_id"].split("_")[0] for m in metadata)
print(f"\nSource breakdown: {dict(sources)}")

In [ ]:
# ── Build or load FAISS index ──────────────────────────────────────────────────
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

INDEX_FILE = VECTOR_DIR / "faiss.index"
TEXT_FILE  = VECTOR_DIR / "chunk_texts.pkl"
META_FILE  = VECTOR_DIR / "metadata.pkl"

if INDEX_FILE.exists():
    # Load pre-built index
    faiss_index = faiss.read_index(str(INDEX_FILE))
    with open(TEXT_FILE, "rb") as f: texts = pickle.load(f)
    with open(META_FILE, "rb") as f: metadata = pickle.load(f)
    print(f"Loaded existing FAISS index: {faiss_index.ntotal} vectors")
else:
    # Build index from scratch
    print("Building FAISS index (this takes ~1 min)...")
    VECTOR_DIR.mkdir(exist_ok=True)
    embeddings = EMBED_MODEL.encode(texts, show_progress_bar=True,
                                    convert_to_numpy=True).astype("float32")
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    faiss.write_index(faiss_index, str(INDEX_FILE))
    with open(TEXT_FILE, "wb") as f: pickle.dump(texts, f)
    with open(META_FILE, "wb") as f: pickle.dump(metadata, f)
    print(f"Index built and saved: {faiss_index.ntotal} vectors")

def vector_retrieve(query, top_k=5):
    """Return top-k text chunks for a query."""
    q_emb = EMBED_MODEL.encode([query], convert_to_numpy=True).astype("float32")
    dists, idxs = faiss_index.search(q_emb, top_k)
    return [texts[i] for i in idxs[0] if i != -1]

def vector_get_context(query, top_k=5):
    return "\n\n".join(vector_retrieve(query, top_k))

# Quick test
sample = vector_retrieve("What causes Alzheimer's disease?", top_k=2)
print(f"\nSample retrieval (top-2 chunks):")
for i, chunk in enumerate(sample):
    print(f"  [{i+1}] {chunk[:120]}...")

## 5 — GraphRAG Pipeline

GraphRAG retrieves a **structured subgraph** relevant to the query rather than raw text passages. For each query, we:
1. Match query keywords against node names and descriptions (simple string overlap)
2. For each matched node, collect its 1-hop neighborhood (entity + type + description + all relations)
3. Serialize the subgraph as text context

In [ ]:
# ── Graph is already built as G above ─────────────────────────────────────────

def graph_find_nodes(query):
    """Return node IDs whose name or description contains any query word."""
    q = query.lower()
    return [nid for nid, data in G.nodes(data=True)
            if q in str(data.get("name","")).lower()
            or q in str(data.get("description","")).lower()]

def graph_get_context(query):
    """Build text context from 1-hop neighborhoods of matched nodes."""
    matched = graph_find_nodes(query)
    if not matched:
        return ""
    context = ""
    visited = set()
    for node in matched:
        if node in visited:
            continue
        visited.add(node)
        nd = G.nodes[node]
        context += f"Entity: {nd.get('name','')}\n"
        context += f"Type  : {nd.get('label','')}\n"
        context += f"Desc  : {nd.get('description','')}\n"
        context += "Relations:\n"
        for nbr in G.neighbors(node):
            e = G.get_edge_data(node, nbr)
            nbr_name = G.nodes[nbr].get("name", nbr)
            context += f"  -- {e.get('relation','')} --> {nbr_name}\n"
        context += "\n"
    return context

# Quick test
sample_ctx = graph_get_context("APOE")
print("Sample GraphRAG context for 'APOE':")
print(sample_ctx[:500] + "...")

## 6 — HybridRAG + Evaluation

### 6.1 Gemini LLM Setup

We use **Gemini Flash Lite** via the Google GenAI API. The free tier allows 15 requests/minute, so we add a 4-second delay between calls and exponential backoff on 429 errors.

In [ ]:
# ── Gemini client ──────────────────────────────────────────────────────────────
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL  = "gemini-flash-lite-latest"
REQUEST_DELAY = 4   # seconds between calls (free tier: 15 req/min)

def llm_generate(question, context, max_retries=5):
    """Generate an answer from the LLM, with rate-limit retry."""
    prompt = f"""You are a medical assistant specializing in Alzheimer's disease.

Answer ONLY using the provided context.
If the answer is not in the context, say: "I could not find the answer in the provided medical knowledge."

-------------------------
Context
-------------------------
{context}

-------------------------
Question
-------------------------
{question}

-------------------------
Answer
-------------------------"""

    for attempt in range(max_retries):
        try:
            time.sleep(REQUEST_DELAY)
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL, contents=prompt)
            return response.text
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = (attempt + 1) * 20
                print(f"  [Rate limit] Waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise e
    raise RuntimeError("Max retries exceeded")

# Test
test_ans = llm_generate("What is Alzheimer's disease?",
                        "Alzheimer's disease is a progressive neurodegenerative disorder.")
print("LLM test response:", test_ans[:200])

### 6.2 Evaluation Metrics

Each generated answer is scored against the gold `expected_answer` using:
- **ROUGE-L F1** — measures longest common subsequence overlap (lexical similarity)
- **Semantic Similarity** — cosine similarity between `all-MiniLM-L6-v2` embeddings (semantic similarity)
- **Exact Match** — strict string equality (mostly 0 for free-text answers, included for completeness)

In [ ]:
# ── Scoring functions ──────────────────────────────────────────────────────────
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score_rouge(expected, predicted):
    return rouge.score(expected, predicted)["rougeL"].fmeasure

def score_similarity(expected, predicted):
    e1 = EMBED_MODEL.encode([expected], convert_to_numpy=True)
    e2 = EMBED_MODEL.encode([predicted], convert_to_numpy=True)
    return float(cosine_similarity(e1, e2)[0][0])

def score_exact(expected, predicted):
    return int(expected.strip().lower() == predicted.strip().lower())

print("Scoring functions ready.")

### 6.3 Run All Three Methods

For each of the 30 evaluation questions, we run **VectorRAG**, **GraphRAG**, and **HybridRAG** — 90 total LLM calls. With a 4-second delay between calls, this takes approximately **8–10 minutes**.

> 💡 If you have already run this and have `Results/ablation_results.csv`, skip this cell and load from file in Section 6.4.

In [ ]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
# Skip if already done
ABLATION_CSV = RESULTS_DIR / "ablation_results.csv"

if ABLATION_CSV.exists():
    print(f"Found existing results at {ABLATION_CSV}")
    print("Loading from file... (delete the CSV to re-run)")
    df_results = pd.read_csv(ABLATION_CSV)
else:
    results = []
    total   = len(questions)

    print(f"Running evaluation on {total} questions × 3 methods = {total*3} LLM calls")
    print(f"Estimated time: ~{total * 3 * 5 // 60} minutes\n")

    for i, item in enumerate(questions):
        qid      = item["question_id"]
        question = item["question"]
        expected = item["expected_answer"]
        favors   = item["designed_to_favor"]

        print(f"[{i+1}/{total}] {qid}: {question[:70]}...")

        # ── VectorRAG ──────────────────────────────────────────────────────────
        t0 = time.perf_counter()
        v_ctx = vector_get_context(question, top_k=5)
        v_ret = time.perf_counter() - t0

        t1 = time.perf_counter()
        v_ans = llm_generate(question, v_ctx)
        v_gen = time.perf_counter() - t1

        # ── GraphRAG ───────────────────────────────────────────────────────────
        t0 = time.perf_counter()
        g_ctx = graph_get_context(question)
        g_ret = time.perf_counter() - t0

        t1 = time.perf_counter()
        g_ans = llm_generate(question, g_ctx)
        g_gen = time.perf_counter() - t1

        # ── HybridRAG (both contexts merged) ───────────────────────────────────
        hybrid_ctx = (
            "========================\nTEXTBOOK KNOWLEDGE\n========================\n"
            + v_ctx
            + "\n\n========================\nKNOWLEDGE GRAPH\n========================\n"
            + g_ctx
        )
        t0 = time.perf_counter()
        h_ans = llm_generate(question, hybrid_ctx)
        h_gen = time.perf_counter() - t0

        # ── Score all three ────────────────────────────────────────────────────
        for method, ans, ret_t, gen_t, ctx in [
            ("VectorRAG",  v_ans, v_ret, v_gen, v_ctx),
            ("GraphRAG",   g_ans, g_ret, g_gen, g_ctx),
            ("HybridRAG",  h_ans, 0,     h_gen, hybrid_ctx),
        ]:
            results.append({
                "question_id":      qid,
                "question":         question,
                "designed_to_favor": favors,
                "method":           method,
                "expected_answer":  expected,
                "generated_answer": ans,
                "rouge_l":          score_rouge(expected, ans),
                "similarity":       score_similarity(expected, ans),
                "exact_match":      score_exact(expected, ans),
                "retrieval_time":   ret_t,
                "generation_time":  gen_t,
                "total_time":       ret_t + gen_t,
            })

        # Quick progress
        row = results[-3:]  # last 3 (one per method)
        for r in row:
            print(f"  {r['method']:<12} similarity={r['similarity']:.3f}  rouge={r['rouge_l']:.3f}")

    df_results = pd.DataFrame(results)
    df_results.to_csv(ABLATION_CSV, index=False, encoding="utf-8")
    print(f"\nSaved: {ABLATION_CSV}")

print(f"\nTotal rows in results: {len(df_results)}")
df_results.head(6)

### 6.4 Results Summary

In [ ]:
# ── Load results if needed ─────────────────────────────────────────────────────
if "df_results" not in dir():
    df_results = pd.read_csv(RESULTS_DIR / "ablation_results.csv")

# ── Summary table ──────────────────────────────────────────────────────────────
summary = (
    df_results.groupby("method")[["similarity", "rouge_l", "exact_match",
                                   "retrieval_time", "generation_time", "total_time"]]
    .mean()
    .round(4)
    .reset_index()
)
summary.to_csv(RESULTS_DIR / "summary_table.csv", index=False)
print("Overall mean scores per method:\n")
print(summary.to_string(index=False))

### 6.5 Visualizations

In [ ]:
# ── Plot 1: Comparison bar chart ───────────────────────────────────────────────
METHODS = ["VectorRAG", "GraphRAG", "HybridRAG"]
COLORS  = ["#4C9BE8", "#E8804C", "#4CE8A0"]

sim_vals   = [df_results[df_results.method==m]["similarity"].mean() for m in METHODS]
rouge_vals = [df_results[df_results.method==m]["rouge_l"].mean()    for m in METHODS]
time_vals  = [df_results[df_results.method==m]["total_time"].mean() for m in METHODS]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("AlzRAGBench — Medical Textbook: RAG Method Comparison",
             fontsize=14, fontweight="bold", y=1.03)

for ax, vals, title, ylabel, ylim in [
    (axes[0], sim_vals,   "Semantic Similarity\n(higher is better)",  "Mean Cosine Sim", (0, 1)),
    (axes[1], rouge_vals, "ROUGE-L F1\n(higher is better)",            "Mean ROUGE-L",    (0, 1)),
    (axes[2], time_vals,  "Avg Latency / Question\n(lower is better)", "Seconds",
     (0, max(time_vals)*1.3)),
]:
    bars = ax.bar(METHODS, vals, color=COLORS, edgecolor="white", linewidth=1.2, width=0.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(*ylim)
    ax.spines[["top","right"]].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+ylim[1]*0.01,
                f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Results/ablation_results.png")

In [ ]:
# ── Plot 2: Per-question ROUGE-L line chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

markers = {"VectorRAG": "o", "GraphRAG": "s", "HybridRAG": "^"}

for method, color, marker in zip(METHODS, COLORS, ["o","s","^"]):
    df_m = df_results[df_results.method == method].reset_index(drop=True)
    ax.plot(range(1, len(df_m)+1), df_m["rouge_l"],
            marker=marker, label=method, color=color,
            linewidth=1.8, markersize=5, alpha=0.85)

ax.set_title("Per-Question ROUGE-L Score by Method", fontsize=13, fontweight="bold")
ax.set_xlabel("Question Index", fontsize=11)
ax.set_ylabel("ROUGE-L F1", fontsize=11)
ax.set_ylim(0, 1); ax.legend(fontsize=11)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_question_rouge.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Results/per_question_rouge.png")

In [ ]:
# ── Plot 3: Score by question bucket (designed_to_favor) ──────────────────────
# This checks: does each method actually win on the question type it was designed for?
bucket_summary = (
    df_results.groupby(["designed_to_favor", "method"])["rouge_l"]
    .mean().unstack("method").reindex(columns=METHODS)
)

fig, ax = plt.subplots(figsize=(10, 5))
bucket_summary.plot(kind="bar", ax=ax, color=COLORS, edgecolor="white",
                    rot=0, width=0.7)
ax.set_title("Mean ROUGE-L by Question Bucket × Method",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Question Bucket (designed_to_favor)", fontsize=11)
ax.set_ylabel("Mean ROUGE-L", fontsize=11)
ax.set_ylim(0, 1)
ax.spines[["top","right"]].set_visible(False)
ax.legend(title="Method", fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "bucket_rouge.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: Results/bucket_rouge.png")

### 6.6 Key Findings

*(Update this cell after running the evaluation with your actual numbers)*

The results are summarized in `Results/ablation_results.png`. Key observations:

- **HybridRAG** combines textbook chunk retrieval with structured graph context, giving the LLM both dense passage-level evidence and entity-relation structure.
- **VectorRAG** benefits from the rich textbook corpus (StatPearls), which provides detailed clinical descriptions not present in abstract-only datasets.
- **GraphRAG** excels on multi-hop questions (`designed_to_favor=graphrag`) where the answer requires tracing relationships between entities — e.g., "how does APOE genotype relate to amyloid clearance and tau pathology?"

The bucket analysis in `Results/bucket_rouge.png` shows whether each method wins on its designed question type — a sharper test than overall averages alone.